# Building a Text Embedding Model from Scratch

Your Java app uses `BgeSmallEnV15QuantizedEmbeddingModel` — a BERT-based model that turns text into
384-dimensional vectors, trained with **contrastive learning** so that semantically similar texts
land near each other in vector space.

This notebook builds that same kind of model from the ground up:

1. **Tokenization** — how text becomes numbers
2. **Transformer encoder** — the architecture that learns contextual representations
3. **Pooling** — collapsing token embeddings into a single sentence vector
4. **Contrastive training** — teaching the model that similar pairs should be close
5. **Evaluation** — measuring retrieval quality

### Prerequisites
```bash
pip install torch transformers datasets sentence-transformers matplotlib
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math
import matplotlib.pyplot as plt
import numpy as np

## Part 1: The Transformer Encoder (from scratch)

BGE-small is a 6-layer BERT with hidden size 384 and 12 attention heads.
Let's build each piece.

### 1.1 Multi-Head Self-Attention

Self-attention lets each token look at every other token to build context-aware representations.
The key insight: the word "bank" should have a different embedding in "river bank" vs "bank account".

For each token, we compute:
- **Query (Q)**: "What am I looking for?"
- **Key (K)**: "What do I contain?"
- **Value (V)**: "What information do I provide?"

Attention(Q, K, V) = softmax(QK^T / √d_k) V

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        # Three linear projections: one each for Q, K, V
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape

        # Project to Q, K, V and reshape for multi-head
        # (batch, seq, hidden) -> (batch, heads, seq, head_dim)
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Mask out padding tokens so they don't influence attention
        if attention_mask is not None:
            # attention_mask: (batch, seq) -> (batch, 1, 1, seq)
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum of values
        context = torch.matmul(attn_weights, v)

        # Concatenate heads and project back
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.out_proj(context)

### 1.2 Transformer Encoder Block

Each block: Self-Attention → Add & LayerNorm → Feed-Forward → Add & LayerNorm

The feed-forward network expands the representation to 4× the hidden size, applies GELU
activation, then compresses back. This is where much of the model's capacity lives.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.attention = MultiHeadSelfAttention(hidden_size, num_heads, dropout)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, hidden_size),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        # Pre-norm variant (slightly easier to train than post-norm)
        residual = x
        x = self.norm1(x)
        x = residual + self.dropout(self.attention(x, attention_mask))

        residual = x
        x = self.norm2(x)
        x = residual + self.ff(x)
        return x

### 1.3 The Full Embedding Model

Assembles: Token Embeddings + Position Embeddings → N Transformer Blocks → Pooling

BGE-small uses **CLS pooling** (take the [CLS] token's output), but mean pooling
(average all non-padding tokens) often works better for sentence embeddings, so we support both.

In [ ]:
class TextEmbeddingModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        hidden_size: int = 384,       # BGE-small uses 384
        num_layers: int = 6,          # BGE-small uses 6 (vs BERT-base's 12)
        num_heads: int = 12,
        ff_dim: int = 1536,           # 4 × hidden_size
        max_seq_len: int = 512,
        dropout: float = 0.1,
        pooling: str = "cls",         # "cls" or "mean"
    ):
        super().__init__()
        self.pooling = pooling

        # Learned embeddings for each token in the vocabulary
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size)
        # Learned position embeddings (BERT-style, not sinusoidal)
        self.position_embeddings = nn.Embedding(max_seq_len, hidden_size)
        self.embed_dropout = nn.Dropout(dropout)

        # Stack of transformer encoder blocks
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(hidden_size, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)

        # Combine token + position embeddings
        x = self.token_embeddings(input_ids) + self.position_embeddings(positions)
        x = self.embed_dropout(x)

        # Pass through each transformer block
        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.final_norm(x)

        # Pool token-level representations into a single sentence vector
        if self.pooling == "cls":
            sentence_embedding = x[:, 0]  # First token ([CLS])
        else:
            # Mean pooling: average over non-padding tokens
            mask = attention_mask.unsqueeze(-1).float()
            sentence_embedding = (x * mask).sum(dim=1) / mask.sum(dim=1)

        # L2 normalize so cosine similarity = dot product
        return F.normalize(sentence_embedding, p=2, dim=-1)

    @property
    def embedding_dim(self):
        return self.token_embeddings.embedding_dim

### Quick sanity check — count parameters

In [ ]:
# Use a small vocab for now; we'll use a real tokenizer later
demo_model = TextEmbeddingModel(vocab_size=30522)  # BERT's vocab size
total_params = sum(p.numel() for p in demo_model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"BGE-small-en-v1.5 has ~33.4M parameters for comparison")

# Test forward pass
dummy_ids = torch.randint(0, 30522, (2, 16))
dummy_mask = torch.ones(2, 16, dtype=torch.long)
out = demo_model(dummy_ids, dummy_mask)
print(f"\nOutput shape: {out.shape}  (batch=2, embedding_dim=384)")
print(f"L2 norms: {torch.norm(out, dim=-1)}  (should be ~1.0)")
del demo_model

---
## Part 2: Contrastive Training

The architecture above produces embeddings, but an untrained model outputs random vectors.
We need to **teach** it that:
- "What programming languages do you know?" and "List your technical skills" should be **close**
- "What programming languages do you know?" and "Where did you go to college?" should be **far apart**

### The InfoNCE / Contrastive Loss

Given a batch of (query, positive_passage) pairs:
1. Embed all queries and passages
2. Compute similarity between every query and every passage
3. Each query's positive passage is the "correct answer"; all other passages are negatives
4. Cross-entropy loss: push the positive similarity up, negatives down

This is exactly how BGE, E5, GTE, and most modern embedding models are trained.

In [ ]:
class InfoNCELoss(nn.Module):
    """Contrastive loss using in-batch negatives.

    Given N (query, passage) pairs, each query has 1 positive and N-1 negatives.
    temperature controls sharpness — lower = harder contrast.
    """
    def __init__(self, temperature: float = 0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, query_embeds, passage_embeds):
        # Similarity matrix: (batch, batch)
        similarity = torch.matmul(query_embeds, passage_embeds.T) / self.temperature

        # Labels: query[i] matches passage[i] (diagonal)
        labels = torch.arange(similarity.shape[0], device=similarity.device)

        return F.cross_entropy(similarity, labels)

### Visualizing how contrastive loss works

In [ ]:
# Simulate a batch of 6 query-passage pairs
torch.manual_seed(42)
q = F.normalize(torch.randn(6, 384), dim=-1)
p = F.normalize(torch.randn(6, 384), dim=-1)

sim = torch.matmul(q, p.T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before training: random similarities, diagonal isn't special
im = axes[0].imshow(sim.detach().numpy(), cmap='RdYlGn', vmin=-1, vmax=1)
axes[0].set_title('Before Training (random)')
axes[0].set_xlabel('Passages')
axes[0].set_ylabel('Queries')

# After training: diagonal should be bright (high similarity)
# Simulate this with identity-like matrix
trained_sim = torch.eye(6) * 0.8 + torch.randn(6, 6) * 0.05
axes[1].imshow(trained_sim.numpy(), cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('After Training (goal)')
axes[1].set_xlabel('Passages')
axes[1].set_ylabel('Queries')

fig.colorbar(im, ax=axes, label='Cosine Similarity')
plt.suptitle('Contrastive Learning: Query-Passage Similarity Matrix')
plt.tight_layout()
plt.show()

---
## Part 3: Dataset — MS MARCO

### Recommended training datasets

| Dataset | Size | Task | Why it's good |
|---------|------|------|---------------|
| **MS MARCO Passages** | 8.8M passages, 500K queries | Search/retrieval | Gold standard for training retrieval models. Real Bing queries + passages. |
| **NLI (SNLI + MultiNLI)** | ~940K pairs | Entailment/contradiction | Great for learning semantic similarity. Many SBERT models start here. |
| **Natural Questions** | 300K+ | Question answering | Real Google queries matched to Wikipedia passages. |
| **HotpotQA** | 113K | Multi-hop QA | Teaches multi-step reasoning similarity. |

**For your resume RAG use case**, MS MARCO is the best choice because it teaches the model to match
short queries ("Python experience") to longer passages (resume sections) — exactly what your app does.

BGE itself was trained on a mix of these plus proprietary data.

Let's load a subset of MS MARCO:

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load MS MARCO passage ranking triples (query, positive, negative)
# Using the community-hosted version with pre-built triples
dataset = load_dataset(
    "sentence-transformers/msmarco-msmarco-distilbert-base-v4",
    split="train",
    streaming=True,  # Stream to avoid downloading the full 1.6GB
)

# Peek at a few examples
examples = list(dataset.take(3))
for i, ex in enumerate(examples):
    print(f"\n--- Example {i+1} ---")
    for key in ex:
        val = str(ex[key])[:120]
        print(f"  {key}: {val}")

If the above dataset isn't available, here's a fallback using NLI data which is smaller and always accessible:

In [ ]:
# Fallback: NLI pairs (always available, smaller download)
try:
    nli = load_dataset("sentence-transformers/all-nli", "pair-score", split="train[:10000]")
    print(f"Loaded {len(nli)} NLI training pairs")
    print(f"Columns: {nli.column_names}")
    print(f"\nExample: {nli[0]}")
except Exception as e:
    print(f"Could not load NLI either: {e}")
    print("You can manually download from: https://huggingface.co/datasets/sentence-transformers/all-nli")

### Tokenizer

We'll use BERT's WordPiece tokenizer. This splits text into subword tokens:
- "embedding" → ["em", "##bed", "##ding"]
- Common words stay whole, rare words get split

This gives us a fixed vocabulary (30,522 tokens) that can handle any English text.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# See how tokenization works
sample = "Senior software engineer with 5 years of Python experience"
tokens = tokenizer.tokenize(sample)
ids = tokenizer.encode(sample)

print(f"Original: {sample}")
print(f"Tokens:   {tokens}")
print(f"IDs:      {ids}")
print(f"Vocab size: {tokenizer.vocab_size}")

---
## Part 4: Training Loop

Now we put it all together. This is a simplified training loop that demonstrates
the core process. Production training would add:
- Hard negative mining (find the most confusing negatives)
- Gradient accumulation (simulate larger batches)
- Warmup + cosine learning rate schedule
- Mixed precision (fp16) training
- Multi-GPU / distributed training

In [ ]:
def tokenize_batch(texts, tokenizer, max_length=128):
    """Tokenize a list of strings into padded tensors."""
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    return encoded["input_ids"], encoded["attention_mask"]


def train_embedding_model(
    model,
    tokenizer,
    train_pairs,       # List of (query, positive_passage) tuples
    num_epochs=3,
    batch_size=32,
    lr=2e-5,
    temperature=0.05,
    device="cpu",
):
    model = model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = InfoNCELoss(temperature=temperature)
    losses = []

    for epoch in range(num_epochs):
        epoch_losses = []

        # Shuffle and batch
        indices = torch.randperm(len(train_pairs))
        for start in range(0, len(train_pairs), batch_size):
            batch_idx = indices[start:start + batch_size]
            queries = [train_pairs[i][0] for i in batch_idx]
            passages = [train_pairs[i][1] for i in batch_idx]

            # Tokenize
            q_ids, q_mask = tokenize_batch(queries, tokenizer)
            p_ids, p_mask = tokenize_batch(passages, tokenizer)
            q_ids, q_mask = q_ids.to(device), q_mask.to(device)
            p_ids, p_mask = p_ids.to(device), p_mask.to(device)

            # Forward pass — encode queries and passages separately
            q_embeds = model(q_ids, q_mask)
            p_embeds = model(p_ids, p_mask)

            # Compute contrastive loss
            loss = loss_fn(q_embeds, p_embeds)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_losses.append(loss.item())

        avg_loss = np.mean(epoch_losses)
        losses.extend(epoch_losses)
        print(f"Epoch {epoch+1}/{num_epochs} — avg loss: {avg_loss:.4f}")

    return losses

### Train on a small demo set

To keep things runnable on a laptop, let's use a tiny set of resume-relevant pairs.
In practice you'd use tens of thousands from MS MARCO.

In [ ]:
# Pairs that are relevant to your resume RAG use case
demo_pairs = [
    ("What programming languages do you know?", "Proficient in Python, Java, and TypeScript with 5 years of experience."),
    ("Tell me about your education", "B.S. in Computer Science from State University, graduated 2019."),
    ("What's your work experience?", "Software Engineer at TechCorp from 2019 to 2023, leading backend development."),
    ("Do you have cloud experience?", "Extensive experience with AWS including EC2, S3, Lambda, and CloudFormation."),
    ("Machine learning skills", "Built and deployed ML models using PyTorch, scikit-learn, and TensorFlow."),
    ("Database experience", "Designed and optimized PostgreSQL and MongoDB schemas for high-throughput applications."),
    ("Leadership experience", "Led a team of 4 engineers to deliver a microservices migration on schedule."),
    ("Open source contributions", "Active contributor to Apache Kafka and maintainer of two popular Python packages."),
    ("Tell me about your projects", "Built a real-time analytics dashboard processing 1M events per second."),
    ("Communication skills", "Presented technical designs to stakeholders and wrote documentation for API consumers."),
    ("DevOps experience", "Set up CI/CD pipelines with GitHub Actions and managed Kubernetes clusters."),
    ("What are your strengths?", "Strong problem-solving skills with a focus on scalable, maintainable architectures."),
    ("Agile experience", "Practiced Scrum for 3 years including sprint planning, retrospectives, and daily standups."),
    ("Frontend skills", "Built responsive UIs with React and Next.js, including component libraries."),
    ("API development", "Designed RESTful APIs and GraphQL endpoints serving 10K requests per second."),
    ("Security experience", "Implemented OAuth2 authentication and conducted security audits on web applications."),
]

print(f"Training on {len(demo_pairs)} pairs")
print(f"In production, BGE-small was trained on millions of pairs.")

In [ ]:
# Create model and train
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

model = TextEmbeddingModel(
    vocab_size=tokenizer.vocab_size,
    hidden_size=384,
    num_layers=6,
    num_heads=12,
    pooling="cls",
)

losses = train_embedding_model(
    model, tokenizer, demo_pairs,
    num_epochs=20,    # More epochs since tiny dataset
    batch_size=8,
    lr=5e-4,          # Higher LR since training from scratch
    device=device,
)

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('InfoNCE Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 5: Evaluation — Does it Actually Work?

Let's test whether our trained model can match queries to the right passages.

In [ ]:
def embed_text(model, tokenizer, text, device="cpu"):
    """Embed a single string."""
    model.eval()
    ids, mask = tokenize_batch([text], tokenizer)
    with torch.no_grad():
        embedding = model(ids.to(device), mask.to(device))
    return embedding.cpu()


def search(query, passages, model, tokenizer, top_k=3, device="cpu"):
    """Find the most similar passages for a query."""
    q_emb = embed_text(model, tokenizer, query, device)

    scores = []
    for passage in passages:
        p_emb = embed_text(model, tokenizer, passage, device)
        score = torch.matmul(q_emb, p_emb.T).item()
        scores.append(score)

    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


# Test retrieval
passages = [p for _, p in demo_pairs]
test_queries = [
    "What cloud platforms have you used?",
    "Tell me about your team leadership",
    "Python experience",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = search(query, passages, model, tokenizer, top_k=3, device=device)
    for rank, (idx, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.3f}] {passages[idx][:80]}...")

---
## Part 6: Compare with the Real BGE Model

Let's see how our from-scratch model compares to the actual BGE-small-en-v1.5
that your Java app uses.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the same model your Java app uses
bge_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print(f"BGE-small architecture:")
print(f"  Parameters: {sum(p.numel() for p in bge_model.parameters()):,}")
print(f"  Embedding dim: {bge_model.get_sentence_embedding_dimension()}")
print(f"  Max seq length: {bge_model.max_seq_length}")

print(f"\nOur model:")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Embedding dim: {model.embedding_dim}")

In [ ]:
# Compare retrieval quality side-by-side
for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")

    # Our model
    print(f"\n  Our model (trained on 16 pairs):")
    results = search(query, passages, model, tokenizer, top_k=3, device=device)
    for rank, (idx, score) in enumerate(results, 1):
        print(f"    {rank}. [{score:.3f}] {passages[idx][:70]}")

    # BGE-small
    print(f"\n  BGE-small-en-v1.5 (trained on millions of pairs):")
    q_emb = bge_model.encode([f"Represent this sentence for searching relevant passages: {query}"])
    p_embs = bge_model.encode(passages)
    scores = q_emb @ p_embs.T
    ranked = sorted(enumerate(scores[0]), key=lambda x: x[1], reverse=True)[:3]
    for rank, (idx, score) in enumerate(ranked, 1):
        print(f"    {rank}. [{score:.3f}] {passages[idx][:70]}")

---
## Key Takeaways

### What we built
- A **transformer encoder** (the same architecture as BERT/BGE)
- Trained with **InfoNCE contrastive loss** (same approach as BGE, E5, GTE)
- Produces **384-dimensional L2-normalized embeddings** (same as BGE-small)

### Why BGE-small is better than our model
1. **Pre-training**: BGE starts from a BERT checkpoint that was pre-trained on massive text corpora
   with masked language modeling (predict missing words). This gives it deep language understanding
   before any contrastive fine-tuning.
2. **Data scale**: Trained on millions of diverse query-passage pairs, not 16.
3. **Hard negative mining**: Uses carefully selected hard negatives (passages that are similar but wrong)
   rather than just random in-batch negatives.
4. **Instruction tuning**: BGE prepends task-specific instructions like
   "Represent this sentence for searching relevant passages:" to improve retrieval.
5. **Distillation**: Knowledge distilled from larger teacher models.

### Recommended next steps to improve this model
1. **Start from a pre-trained BERT** — replace random init with `AutoModel.from_pretrained("bert-base-uncased")`
2. **Train on full MS MARCO** — 500K+ query-passage pairs
3. **Add hard negatives** — use BM25 to find challenging negatives
4. **Multi-task training** — mix retrieval, NLI, and STS datasets

### Connection to your Java app
Your `ResumeEmbeddingProducer` loads the quantized ONNX version of this same architecture.
The ONNX export + quantization (INT8) makes it ~4× smaller and faster while keeping
~99% of the retrieval quality — important for running in a Quarkus app without a GPU.